In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
df = yf.download("^FCHI", start="2015-01-01", end="2026-10-01")
df.head()

C:\Users\tiago\quant-alpha\.venv\Lib\site-packages\yfinance\utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
C:\Users\tiago\quant-alpha\.venv\Lib\site-packages\yfinance\utils.py:514: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(
C:\Users\tiago\quant-alpha\.venv\Lib\site-packages\yfinance\utils.py:518: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.rename(columns={
[*********************100%***********************]  1 o

Price,Close,High,Low,Open,Volume
Ticker,^FCHI,^FCHI,^FCHI,^FCHI,^FCHI
Date,,,,,
2015-01-02,4252.290039,4311.000000,4224.339844,4294.049805,69809300
2015-01-05,4111.359863,4276.919922,4105.450195,4221.990234,137887700
2015-01-06,4083.500000,4151.410156,4076.159912,4129.890137,130814400
2015-01-07,4112.729980,4144.950195,4080.780029,4111.729980,121316600
2015-01-08,4260.189941,4270.109863,4163.629883,4176.160156,154417100


In [2]:
df.columns = df.columns.droplevel("Ticker")

In [3]:
print(df.columns)
print(df.shape)
print(df.isna().sum())

Index(['Close', 'High', 'Low', 'Open', 'Volume'], dtype='str', name='Price')
(3006, 5)
Price
Close     0
High      0
Low       0
Open      0
Volume    0
dtype: int64


In [4]:
df["ret"] = np.log(df["Close"]).diff()
df["ret"].describe()

count    3005.000000
mean        0.000209
std         0.011539
min        -0.130983
25%        -0.005005
50%         0.000580
75%         0.006082
max         0.080561
Name: ret, dtype: float64

In [5]:
print(df["ret"].idxmin(), df["ret"].idxmax())

2020-03-12 00:00:00 2020-03-24 00:00:00


In [6]:
df.to_csv("cac40_prices.csv")

In [7]:
data = df.copy()
data["ret_next"] = data["ret"].shift(-1)
data = data.dropna(subset = ["ret_next"])
data["target"] = (data["ret_next"] > 0).astype(int)
data["target"]

Date
2015-01-02    0
2015-01-05    0
2015-01-06    1
2015-01-07    1
2015-01-08    0
             ..
2026-09-23    0
2026-09-24    0
2026-09-25    1
2026-09-28    0
2026-09-29    0
Name: target, Length: 3005, dtype: int64

In [8]:
print(data.shape)
print(data["target"].mean())

(3005, 8)
0.5254575707154742


In [9]:
print(data.index[0], data.index[-1])
print(data["ret"].isna().sum())
print(data["ret_next"].isna().sum())

2015-01-02 00:00:00 2026-09-29 00:00:00
1
0


In [10]:
print(len(data), len(df))
print(data.index[-1])

3005 3006
2026-09-29 00:00:00


In [11]:
data["ret_lag1"] = data["ret"].shift(1)

In [12]:
data[["ret", "ret_lag1"]].head()

Price,ret,ret_lag1
Date,,
2015-01-02,NaN,NaN
2015-01-05,-0.033704,NaN
2015-01-06,-0.006799,-0.033704
2015-01-07,0.007133,-0.006799
2015-01-08,0.035227,0.007133


In [13]:
data["ret_lag2"] = data["ret"].shift(2)
data["vol20"] = data["ret"].rolling(20).std()
print(data[["ret_lag1", "ret_lag2", "vol20"]].isna().sum())

Price
ret_lag1     2
ret_lag2     3
vol20       20
dtype: int64


In [14]:
vol = data["Volume"].replace(0, np.nan)
data["vol_rel"] = vol / vol.rolling(20, min_periods=10).mean()

In [15]:
print((data["Volume"] == 0).sum())
print(data["vol_rel"].isna().sum())
print(data["Volume"].describe())

74
95
count    3.005000e+03
mean     8.262753e+07
std      3.795625e+07
min      0.000000e+00
25%      6.067690e+07
50%      7.606720e+07
75%      9.774040e+07
max      4.157757e+08
Name: Volume, dtype: float64


In [16]:
zeros = data[data["Volume"] == 0]
print(zeros.index.min() , zeros.index.max())
print(zeros.index.year.value_counts().sort_index())
print(data["vol_rel"].isna().iloc[19:].sum())


2020-10-26 00:00:00 2024-09-10 00:00:00
Date
2020    37
2021    13
2022     8
2023    12
2024     4
Name: count, dtype: int64
86


In [17]:
data[data["vol_rel"].isna()].index[-1]

Timestamp('2024-09-10 00:00:00')

In [18]:
print(vol.isna().sum())

74


In [19]:
mask = data["vol_rel"].isna() & vol.notna()
print(mask.sum())
print(data.index[mask].min(), data.index[mask].max())

21
2015-01-02 00:00:00 2021-01-04 00:00:00


In [20]:
print(data.index[mask])

DatetimeIndex(['2015-01-02', '2015-01-05', '2015-01-06', '2015-01-07',
               '2015-01-08', '2015-01-09', '2015-01-12', '2015-01-13',
               '2015-01-14', '2020-11-23', '2020-11-26', '2020-11-27',
               '2020-12-07', '2020-12-09', '2020-12-11', '2020-12-18',
               '2020-12-21', '2020-12-23', '2020-12-28', '2020-12-30',
               '2021-01-04'],
              dtype='datetime64[s]', name='Date', freq=None)


In [21]:
data["range_pct"] = (data["High"] - data["Low"]) / data["Close"]
data["range_pct"].isna().sum()

np.int64(0)

In [22]:
features = ["ret", "ret_lag1", "ret_lag2", "vol20", "vol_rel", "range_pct"]
final = data[features + ["target"]].dropna()
print(len(final))

2899


In [23]:
cut = int(len(final) * 0.8)
train = final.iloc[:cut]
test = final.iloc[cut + 20:]
print(len(train), len(test))
print(train.index[-1], test.index[0])


2319 560
2024-06-18 00:00:00 2024-07-19 00:00:00


In [24]:
print(train["target"].mean())
print(test["target"].mean())

0.5282449331608452
0.5071428571428571


In [25]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

X_train = scaler.fit_transform(train[features])
print(X_train)

X_test = scaler.transform(test[features])
print(X_test)

[[-0.51340267  0.36142116 -0.25269998  0.97226154 -0.13854859  0.53207349]
 [ 0.41135908 -0.50510175  0.36206883  0.51513893 -0.53560726  0.12244532]
 [ 0.89418343  0.41513507 -0.50609437  0.46293631 -0.04607677 -0.04995466]
 ...
 [-2.28844136 -1.70053984  0.8004441  -0.07529613  4.18157037  2.13191508]
 [ 0.74784547 -2.27145503 -1.70379539 -0.02354035  0.26672245  0.0702878 ]
 [ 0.61862922  0.74997501 -2.2757913   0.02623133  0.07508198 -0.29170991]]
[[-0.59743263  0.16350126 -0.11332569 -0.25055097  0.15534215 -0.61321415]
 [ 0.95751101 -0.58872054  0.16377428 -0.23516919 -0.50826736 -0.17163969]
 [-0.27609117  0.95861464 -0.58987145 -0.24601646 -0.42156393 -0.09538643]
 ...
 [-0.05466784 -0.44592691 -0.33652002 -0.54046757 -0.55550683 -0.33016642]
 [-0.00964959 -0.04861154 -0.44680751 -0.56262995 -0.60797561 -0.5351789 ]
 [-0.46298839 -0.00381357 -0.04874005 -0.55741077 -0.34431865 -0.25943051]]


In [26]:
y_train = train["target"]
print(y_train)
y_test = test["target"]
print(y_test)

Date
2015-01-30    1
2015-02-02    1
2015-02-03    1
2015-02-04    1
2015-02-05    0
             ..
2024-06-12    0
2024-06-13    0
2024-06-14    1
2024-06-17    1
2024-06-18    0
Name: target, Length: 2319, dtype: int64
Date
2024-07-19    1
2024-07-22    0
2024-07-23    0
2024-07-24    0
2024-07-25    1
             ..
2026-09-23    0
2026-09-24    0
2026-09-25    1
2026-09-28    0
2026-09-29    0
Name: target, Length: 560, dtype: int64


In [27]:
print(X_train.mean(axis=0).round(3))
print(X_train.std(axis=0).round(3))
print(X_test.mean(axis=0).round(3))
print(X_test.std(axis=0).round(3))

[ 0.  0. -0. -0.  0.  0.]
[1. 1. 1. 1. 1. 1.]
[-0.008 -0.003  0.    -0.249  0.003 -0.199]
[0.796 0.792 0.793 0.573 0.998 0.722]


In [28]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(X_train, y_train)
print(model.score(X_test, y_test))
print(model.coef_.round(3))
print(model.intercept_.round(3))
print((model.predict(X_test) == 1).mean())

0.5035714285714286
[[-0.046 -0.011 -0.033  0.099  0.048 -0.07 ]]
[0.113]
0.9571428571428572


In [29]:
train["target"].mean()

np.float64(0.5282449331608452)

In [30]:
proba = model.predict_proba(X_test)[:, 1]
print(proba.min().round(3), proba.max().round(3), proba.mean().round(3))
print((proba > 0.5).mean())

0.471 0.61 0.526
0.9571428571428572
